# Week 13: Reinforcement Learning & RLHF
## Notebook 11: ORPO: Monolithic Odds Ratio Preference Optimization

### 1. Conceptual Intuition & Theoretical Background
Standard alignment methods (like DPO or PPO) require keeping a **reference model** $\pi_{\text{ref}}$ in GPU memory to compute KL-divergence penalties, doubling VRAM requirements. Moreover, standard Supervised Fine-Tuning (SFT) can inadvertently raise the log-probability of rejected responses.

**ORPO (Hong et al., 2024):**
Unifies SFT and preference alignment into a **single, monolithic loss function** without requiring a reference model:
$$\mathcal{L}_{\text{ORPO}} = \mathcal{L}_{\text{SFT}} + \lambda \cdot \mathcal{L}_{\text{OR}}$$

**1. SFT Loss (Negative Log-Likelihood on Chosen Response):**
$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|y_w|} \sum_{t} \ln \pi_\theta(y_{w, t} \mid x, y_{w, <t})$$

**2. Odds Ratio Penalty ($\mathcal{L}_{\text{OR}}$):**
$$\text{Odds}_\theta(y \mid x) = \frac{P_\theta(y \mid x)}{1 - P_\theta(y \mid x)}$$
$$\mathcal{L}_{\text{OR}} = -\ln \sigma\left( \ln \frac{\text{Odds}_\theta(y_w \mid x)}{\text{Odds}_\theta(y_l \mid x)} \right)$$
Penalizes the policy if the odds of generating the rejected response $y_l$ approach the odds of the chosen response $y_w$.

---

### 2. Scratch Python Implementation: ORPO Loss Function

In [ ]:
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))

def orpo_loss_scratch(logp_chosen, logp_rejected, lambda_reg=0.1):
    """
    Computes monolithic ORPO loss without a reference model.
    logp_chosen: Average per-token log prob of chosen response log P(y_w | x)
    logp_rejected: Average per-token log prob of rejected response log P(y_l | x)
    """
    # 1. Standard SFT loss on chosen output
    loss_sft = -np.mean(logp_chosen)
    
    # 2. Convert log-probabilities to Odds: Odds = P / (1 - P)
    # log(Odds) = log(P) - log(1 - P) = logp - log1p(-exp(logp))
    # Approximation for small P: log(Odds) ~ logp
    log_odds_chosen = logp_chosen - np.log(1.0 - np.exp(np.clip(logp_chosen, -50.0, -1e-4)) + 1e-9)
    log_odds_rejected = logp_rejected - np.log(1.0 - np.exp(np.clip(logp_rejected, -50.0, -1e-4)) + 1e-9)
    
    # Log Odds Ratio: log(Odds_w / Odds_l) = log_odds_w - log_odds_l
    log_odds_ratio = log_odds_chosen - log_odds_rejected
    
    # Odds Ratio Loss: -log(sigmoid(log_odds_ratio))
    loss_or = -np.mean(np.log(sigmoid(log_odds_ratio) + 1e-9))
    
    total_loss = loss_sft + lambda_reg * loss_or
    return total_loss, loss_sft, loss_or

# Benchmark on 3 Prompt Scenarios
# Scenario 1: Model strongly prefers chosen (Healthy)
# Scenario 2: Model assigns equal probability to both (Confused)
# Scenario 3: Model prefers rejected output (Pathological)
logp_w = np.array([-0.3, -1.0, -2.5])
logp_l = np.array([-2.8, -1.0, -0.4])

total_l, sft_l, or_l = orpo_loss_scratch(logp_w, logp_l, lambda_reg=0.2)

print("--- ORPO Monolithic Preference Optimization ---")
print(f"Total Combined ORPO Loss: {total_l:.4f}")
print(f"SFT Component:            {sft_l:.4f}")
print(f"Odds Ratio Component:     {or_l:.4f}")
print("ORPO eliminates reference model forward passes, halving training VRAM!")

### 3. Step-by-Step Code Walkthrough
1. **SFT Preservation:** Retains token prediction loss on preferred responses.
2. **Log-Odds Formulation:** Converts sequence probabilities into log-odds ratios.
3. **No Reference Baseline:** Operates without a frozen reference model, saving GPU VRAM.

---

### 4. Industry Applications & Trade-offs
- **Single-Stage LLM Fine-Tuning:** Merging SFT and alignment into a single training step.
- **Trade-off:** Sensitive to sequence-length normalization between chosen and rejected responses.

---

### 5. Top Interview Questions & Answers
**Q1: How does ORPO eliminate the need for a reference model like the one used in DPO?**
> **Answer:** DPO relies on a reference model $\pi_{\text{ref}}$ to prevent the policy from degenerating via a KL-divergence penalty. ORPO directly penalizes the Odds Ratio between chosen and rejected responses within the SFT loss itself.

**Q2: Why can standalone SFT lead to degradation on preference benchmarks?**
> **Answer:** Standard cross-entropy SFT increases the overall generation likelihood of common n-grams, which can inadvertently raise the probabilities of undesirable phrasing present in the training domain.